In [10]:
import yfinance as yf
import datetime as dt
import pandas as pd
import numpy as np
import datetime
import copy
import matplotlib.pyplot as plt

In [36]:
# ----Getting Data----
tickers = ["MMM","AXP","T","BA","CAT","CSCO","KO","XOM","GE","GS","HD","IBM","INTC","JNJ","JPM","MCD","MRK","MSFT","NKE","PFE","PG","TRV","UNH","VZ","V","WMT","DIS"]

ohlc_mon = {}
start = dt.datetime.today() - dt.timedelta(3650)
end = dt.datetime.today()

for ticker in tickers:
    ohlc_mon[ticker] = yf.download(ticker, start, end, interval='1mo')
    ohlc_mon[ticker].dropna(inplace=True, how='all')

tickers = ohlc_mon.keys()


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [55]:
# useful functions 
def cagr(ret_df, ppy=12, col="mon_ret"):
    #df = DF.copy()

    #df["return"] = df["mon_ret"].pct_change()
    #df["cum_return"] = (1 + df["return"]).cumprod()

    r = ret_df[col].dropna()

    # cum return ^ 1/n
    n = len(r)/ppy # 12 months in a year
    # if data in hours, divide n further by 8 (8 hours in trading day)

    CAGR = ((1+r).prod())**(1/n) - 1

    return CAGR

def volatility(ret_df, ppy, col="mon_ret"):
    #df = DF.copy()

    #df["return"] = df["mon_ret"].pct_change()

    r = ret_df[col].dropna()

    # volatility is just st dev of returns
    # annualised volatility is volatility multiplied by sqrt(252)
    vol = r.std() * np.sqrt(12) # 12 months in a year

    return vol

def sharpe(ret_df, rf=0.0, ppy=12, col='mon_ret'):
    #df = DF.copy()
    v = volatility(ret_df, ppy, col)
    return (cagr(ret_df, ppy, col) - rf)/v

def max_dd(ret_df, col='mon_ret'):
    #df = DF.copy()
    #df["return"] = df["mon_ret"].pct_change()
    #df["cum_return"] = 1 + df["return"].cumprod()
    #df["cum_roll_max"] = df["cum_return"].cummax()
    #df["drawdown"] = df["cum_roll_max"] - df["cum_return"]

    r = ret_df[col].dropna()
    equity = (1+r).cumprod()
    peak = equity.cummax()
    dd = (equity/peak) - 1 # neg drawdown

    return dd.min()

In [56]:
# ----Backtesting----

# Always best to first run an initial buy and hold test to place a benchmark for strategy
# if buy and hold over past 10 years give return of 10%, then strategy must outperform this

ohlc_dict = copy.deepcopy(ohlc_mon) # deepcopy ensures original data isn't tampered
return_df = pd.DataFrame()

for ticker in tickers:
    #print("Calculating monthly return for", ticker)
    ohlc_dict[ticker]["mon_ret"] = ohlc_dict[ticker]["Close"].pct_change()
    return_df[ticker] = ohlc_dict[ticker]["mon_ret"]

In [58]:
# calculating portfolio return iteratively
#DF = return_df
#m = 6
#x = 3
#i = 10
portfolio = ["MMM","AXP","T","BA","CAT","CSCO"]

def pfolio(DF, m = 6, x = 3):
    # DF = dataframe with monthly return info for ALL stocks
    # m = num of stocks in portfolio
    # x = num of underperforming stocks to be removed from portfolio monthly
    df = DF.copy()
    portfolio = []
    monthly_ret = [0]

    # loop from 1 to length of df, traverse ever row of dataframe
    for i in range(1, len(df)):
        # only run if something exists in portfolio
        if len(portfolio) > 0:
            # df[portfolio].iloc[i,:] means: getting all the rows which correspond to i in the portfolio and all the columns. [i,:] means i-th row but all its columns. then storing the monthly return by appending
            monthly_ret.append(df[portfolio].iloc[i,:].mean())

            # sorting the portfolio in to ascending order, so top three would be worst performing. 
            bad_stocks = df[portfolio].iloc[i,:].sort_values(ascending=True)[:x].index.values.tolist()

            # removing bad stocks
            portfolio = [t for t in portfolio if t not in bad_stocks]

        # adding the new stocks
        fill = m - len(portfolio)
        # ensuring new stock added is unique
        new_picks = df[[t for t in tickers if t not in portfolio]].iloc[i,:].sort_values(ascending=False)[:fill].index.values.tolist()
        # adding them to portfolio
        portfolio = portfolio + new_picks
        #checking
        #print(portfolio)
    monthly_ret_df = pd.DataFrame(np.array(monthly_ret), columns=["mon_ret"])
    
    return monthly_ret_df

#print("monthly_ret_df:", pfolio(return_df).head())

# strategy's KPIs
print("CAGR gives compounded annual growth rate. CAGR:", cagr(pfolio(return_df)))
print("Sharpe Ratio gives investments risk-adjusted return. Higher is better. Sharpe:", sharpe(pfolio(return_df), 0.025))
print("Max Drawdown measures largest percent decline from portfolio's highest peak to lowest subsequent trough. Max_DD:", max_dd(pfolio(return_df, 6, 3)))


CAGR gives compounded annual growth rate. CAGR: 0.14607495334873666
Sharpe Ratio gives investments risk-adjusted return. Higher is better. Sharpe: 0.7348511790316156
Max Drawdown measures largest percent decline from portfolio's highest peak to lowest subsequent trough. Max_DD: -0.23234999672264212


In [ ]:
# this is an example strategy
